In [2]:
import os
import random
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage

In [3]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError("Please set your GROQ_API_KEY environment variable.")
model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [4]:
# 评分路由系统
def score_based_routing():
    """
    根据评分决定处理流程
    - 优秀 (>= 90)：发送表扬信
    - 良好 (>= 70)：正常通过
    - 需改进 (>= 50)：提供建议
    - 不合格 (< 50)：需要重新提交
    """
    print("\n"+"="*60)
    print("Score Based Routing")
    print("="*60)

    class ScoreState(TypedDict):
        content: str
        score: int
        feedback: str
        result: str

    def evaluate(state: ScoreState) -> dict:
        """评估内容并打分"""
        messages = [
            SystemMessage(content="""你是一个内容评估专家。请评估以下内容并给出1-100的分数。
只返回一个数字分数，不要其他内容。评估标准：
- 90-100：优秀，内容完整、准确、有创意
- 70-89：良好，内容基本完整，表达清晰
- 50-69：需改进，内容有所欠缺
- 0-49：不合格，需要重新撰写"""),
            HumanMessage(content=state["content"])
        ]
        response = model.invoke( messages)
        try:
            score = int(response.content.strip())
            score = max(0, min(100, score)) # 限制分数范围
        except:
            score = 70 # 默认分数
        print(f"[评估] 评分: {score}")
        return {"score": score}

    def route_by_score(state: ScoreState) -> Literal["excellent", "good", "improve", "reject"]:
        """根据分数路由"""
        score = state["score"]
        if score >= 90:
            return "excellent"
        elif score >= 70:
            return "good"
        elif score >= 50:
            return "improve"
        else:
            return "reject"

    def handle_excellent(state: ScoreState) -> dict:
        """处理优秀评分"""
        print("[优秀] 发送表扬通知")
        return {
            "feedback": "恭喜！你的内容非常出色！",
            "result": "APPROVED_WITH_HONORS"
        }

    def handle_good(state: ScoreState) -> dict:
        """处理良好评分"""
        print("  [良好] ✅ 正常通过")
        return {
            "feedback": "内容合格，已通过审核。",
            "result": "APPROVED"
        }

    def handle_improve(state: ScoreState) -> dict:
        """处理需改进评分"""
        print("  [需改进] 📝 生成改进建议")
        messages = [
            SystemMessage(content="请为以下内容提供简洁的改进建议（50字以内）。用中文回复。"),
            HumanMessage(content=state["content"])
        ]
        response = model.invoke(messages)
        return {
            "feedback": f"建议改进：{response.content}",
            "result": "NEEDS_IMPROVEMENT"
        }

    def handle_reject(state: ScoreState) -> dict:
        """处理不合格评分"""
        print("  [不合格] ❌ 需要重新提交")
        return {
            "feedback": "内容不符合要求，请重新撰写并提交。",
            "result": "REJECTED"
        }

    # 构造图
    graph = StateGraph(ScoreState)
    # 添加节点
    graph.add_node("evaluate", evaluate)
    graph.add_node("excellent", handle_excellent)
    graph.add_node("good", handle_good)
    graph.add_node("improve", handle_improve)
    graph.add_node("reject", handle_reject)
    # 添加边
    graph.add_edge(START, "evaluate")
    graph.add_conditional_edges(
        "evaluate",
        route_by_score,
        {
            "excellent": "excellent",
            "good": "good",
            "improve": "improve",
            "reject": "reject"
        }
    )
    for node in ["excellent", "good", "improve", "reject"]:
        graph.add_edge(node, END)
    # 编译
    app = graph.compile()
    # 测试
    test_contents = [
        "Python 是一种广泛使用的高级编程语言，以其清晰的语法和强大的功能著称。它支持多种编程范式，包括面向对象、函数式和过程式编程。Python 拥有丰富的标准库和第三方库，广泛应用于 Web 开发、数据科学、人工智能等领域。",
        "Python 是编程语言，很好用。",
        "编程"
    ]

    for content in test_contents:
        print(f"\n提交内容: {content[:30]}...")
        result = app.invoke({"content": content})
        print(f"结果: {result['result']}")
        print(f"反馈: {result['feedback']}")

In [5]:
# 重试机制
def retry_mechanism():
    """
    实现带重试的工作流
    - 任务可能随机失败
    - 最多重试 3 次
    - 超过重试次数后使用备用方案
    """
    print("\n"+"="*60)
    print("Retry Mechanism")
    print("="*60)

    class RetryState(TypedDict):
        task: str
        retry_count: int
        max_retries: int
        success: bool
        result: str
        error_messages: str

    def execute_task(state: RetryState) -> dict:
        """执行可能失败的任务"""
        retry_count = state.get("retry_count", 0)
        # 模拟任务执行（50％概率失败）
        success = random.random() > 0.5
        if success:
            print(f"  [执行] ✅ 任务成功 (尝试 {retry_count + 1})")
            return {
                "success": True,
                "result": f"任务 '{state['task']}' 执行成功！",
                "retry_count": retry_count + 1
            }
        else:
            print(f"  [执行] ❌ 任务失败 (尝试 {retry_count + 1})")
            return {
                "success": False,
                "error_message": "模拟的随机错误",
                "retry_count": retry_count + 1
            }
    def should_retry(state: RetryState) -> Literal["retry", "fallback", "success"]:
        """决定是否重试"""
        if state["success"]:
            return "success"
        if state["retry_count"] < state["max_retries"]:
            print(f"  [路由] 准备第 {state['retry_count'] + 1} 次重试...")
            return "retry"
        print("  [路由] 重试次数已达上限，使用备用方案")
        return "fallback"
    def success_handler(state: RetryState) -> dict:
        """成功处理"""
        return {"result": f"✅ 最终结果：{state['result']}"}
    def fallback_handler(state: RetryState) -> dict:
        """备用方案"""
        print("  [备用] 执行备用方案...")
        return {
            "result": f"⚠️ 使用备用方案完成任务（原任务失败 {state['retry_count']} 次）"
        }
    # 构建图
    graph = StateGraph(RetryState)
    # 添加节点
    graph.add_node("execute", execute_task)
    graph.add_node("success", success_handler)
    graph.add_node("fallback", fallback_handler)
    # 添加边
    graph.add_edge(START, "execute")
    graph.add_conditional_edges(
        "execute",
        should_retry,
        {
            "retry": "execute",      # 重试：回到执行节点
            "fallback": "fallback",  # 备用方案
            "success": "success"     # 成功
        }
    )
    graph.add_edge("success", END)
    graph.add_edge("fallback", END)
    # 编译
    app = graph.compile()
    # 测试
    for i in range(3):
        print(f"\n--- 测试 {i + 1} ---")
        result = app.invoke({
            "task": "发送通知邮件",
            "retry_count": 0,
            "max_retries": 3,
            "success": False
        })
        print(f"结果: {result['result']}")

In [6]:
def main():
    print("\n"+"="*70)
    print("Conditional Routing")
    print("="*70)

    score_based_routing()
    retry_mechanism()

In [7]:
if __name__ == "__main__":
    main()


Conditional Routing

Score Based Routing

提交内容: Python 是一种广泛使用的高级编程语言，以其清晰的语法和...
[评估] 评分: 85
  [良好] ✅ 正常通过
结果: APPROVED
反馈: 内容合格，已通过审核。

提交内容: Python 是编程语言，很好用。...
[评估] 评分: 10
  [不合格] ❌ 需要重新提交
结果: REJECTED
反馈: 内容不符合要求，请重新撰写并提交。

提交内容: 编程...
[评估] 评分: 50
  [需改进] 📝 生成改进建议
结果: NEEDS_IMPROVEMENT
反馈: 建议改进：增加注释，优化代码结构。

Retry Mechanism

--- 测试 1 ---
  [执行] ✅ 任务成功 (尝试 1)
结果: ✅ 最终结果：任务 '发送通知邮件' 执行成功！

--- 测试 2 ---
  [执行] ✅ 任务成功 (尝试 1)
结果: ✅ 最终结果：任务 '发送通知邮件' 执行成功！

--- 测试 3 ---
  [执行] ❌ 任务失败 (尝试 1)
  [路由] 准备第 2 次重试...
  [执行] ❌ 任务失败 (尝试 2)
  [路由] 准备第 3 次重试...
  [执行] ✅ 任务成功 (尝试 3)
结果: ✅ 最终结果：任务 '发送通知邮件' 执行成功！
